# Preparación de entradas para RF3

## Objetivo

Cada equipo deberá:
1. Seleccionar **dos variantes**;
2. Justificar por qué son informativas;
3. Cargar la proteína WT correcta;
4. Validar que cada variante coincide con el residuo WT esperado;
5. Generar las dos proteínas mutantes;
6. Construir el archivo JSON para RF3.


# Archivos necesarios

### A. Tabla de variantes
Debe contener al menos:
- `ID`
- `protein_change`

Ejemplos:
- `p.Arg408Trp`
- `p.Ser83Leu`

### B. Proteína WT
Un FASTA validado de la proteína de referencia.


In [1]:
from pathlib import Path  # rutas de archivos de forma portable
import json  # para escribir el archivo de entrada de RF3
import re  # expresiones regulares (para leer "p.Arg408Trp")
import pandas as pd  # tabla de variantes

# Muestra todas las columnas al imprimir tablas (sin truncar con "...")
pd.set_option("display.max_columns", None)

# 1. Configuración del equipo

Modifiquen únicamente esta celda.

Ejemplos:

### Caso PAH
- `VARIANT_TABLE = "PAH_pre_vs_Evo2.tsv"`
- `WT_FASTA = "PAH_WT_NP_000268.1.fasta"`
- `PROTEIN_NAME = "PAH"`

### Caso E. coli
- `VARIANT_TABLE = "Ecoli_pre_vs_Evo2.tsv"`
- `WT_FASTA = "GyrA_WT_validado.fasta"`
- `PROTEIN_NAME = "GyrA"`

In [2]:
# ============================================================
# EDITAR ESTA CELDA
# ============================================================

# Nombres de archivo: deben estar subidos a Colab (panel de archivos) o en la carpeta actual
VARIANT_TABLE = "PAH_pre_vs_Evo2.tsv"
WT_FASTA = "PAH_WT_NP_000268.1.fasta"

# Se usa como prefijo en los nombres de los archivos de salida
PROTEIN_NAME = "PAH"

# Nombres de las columnas de la tabla (cámbienlos solo si su TSV usa otros)
ID_COLUMN = "ID"
PROTEIN_CHANGE_COLUMN = "protein_change"

# Identificadores de las dos variantes elegidas (deben existir en la columna ID)
SELECTED_VARIANTS = [
    "VAR007",
    "VAR003",
]

# 2. Cargar archivos

In [4]:
def find_file(name):
    # Detecta automáticamente la carpeta raíz 'Caso_1_PHA_Humano' subiendo niveles
    cwd = Path.cwd()
    repo_root = next((p for p in [cwd, *cwd.parents] if p.name == "Caso_1_PHA_Humano"), cwd)

    candidates = [
        Path(name),                                                  # 1. Carpeta actual
        Path("../../database/protein") / name,                       # 2. Ruta relativa desde resultados/rf3/
        repo_root / "database" / "protein" / name,                   # 3. Desde la raíz del repo (local)
        Path("/content/Caso_1_PHA_Humano/database/protein") / name,  # 4. Ruta típica en Google Colab
        Path("../") / name
    ]

    found = next((p for p in candidates if p.exists()), None)
    if found is None:
        rutas = "\n  - ".join(str(p.resolve()) for p in candidates)
        raise FileNotFoundError(f"No encontré '{name}'. Se buscó en:\n  - {rutas}")
    return found

variant_path = find_file(VARIANT_TABLE)
fasta_path = find_file(WT_FASTA)

# El TSV se lee separado por tabuladores
variants = pd.read_csv(variant_path, sep="\t")

# Vista rápida para comprobar que la tabla se cargó bien
display(variants.head())
print("Tabla:", variant_path)
print("FASTA:", fasta_path)

,ID,protein_change,transcript_change,consequence,dominio,POPAF,BLOSUM62,puntaje_inicial,rank_inicial,delta_sitio_genoma,delta_ventana_genoma,delta_sitio_gen,delta_ventana_gen,rank_evo2,cambio_vs_inicial,seleccion_RF3
0,VAR003,p.Arg408Trp,c.1222C>T,missense,catalítico,0.000920,-3.0,8.0,1,-5.72,-16.06,0.58,-31.65,4.0,-3.0,True
1,VAR005,p.Asp338Ala,c.1013A>C,missense,catalítico,0.000004,-2.0,8.0,2,-4.14,-18.98,-0.20,-29.51,3.0,-1.0,False
2,VAR007,p.Gly247Asp,c.740G>A,missense,catalítico,0.000010,-1.0,7.5,3,-7.83,-22.99,-1.48,-21.58,1.0,2.0,True
3,VAR011,p.Thr117Ile,c.350C>T,missense,regulador,0.000050,-1.0,7.5,4,0.15,-11.50,-5.18,-5.90,5.0,-1.0,False
4,VAR012,p.Ala104_Val106del,c.310_318del,delecion_in_frame,regulador,0.000010,NaN,7.5,5,NaN,NaN,NaN,NaN,NaN,NaN,False


Tabla: ../PAH_pre_vs_Evo2.tsv
FASTA: ../../database/protein/PAH_WT_NP_000268.1.fasta


# 3. Leer proteína WT

El lector FASTA usa sólo librería estándar para no introducir dependencias innecesarias.

In [5]:
def read_fasta(path):
    lines = Path(path).read_text().splitlines()

    # Cada cabecera ">" inicia un registro; la actividad requiere una sola proteína
    n_records = sum(1 for line in lines if line.startswith(">"))
    if n_records > 1:
        raise ValueError(
            f"El FASTA tiene {n_records} registros; usen un FASTA con una sola proteína."
        )

    # Une todas las líneas que no son cabecera (">") en una sola secuencia
    seq = "".join(
        line.strip()
        for line in lines
        if line and not line.startswith(">")
    ).upper()

    if not seq:
        raise ValueError("El FASTA no contiene secuencia.")

    return seq

wt_seq = read_fasta(fasta_path)

# Comprobación visual: nombre, longitud y primeros residuos
print("Proteína:", PROTEIN_NAME)
print("Longitud WT:", len(wt_seq), "aa")
print(wt_seq[:60] + "...")

Proteína: PAH
Longitud WT: 452 aa
MSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDV...


# 4. Selección de candidatos

Elijan exactamente dos variantes.

La selección debe responder una pregunta estructural.

### Justificación del equipo

**Candidato 1:**  
VAR007 (p.Gly247Asp)

**Candidato 2:**  
VAR003 (p.Arg408Trp)

**Pregunta estructural:**  
¿Qué variante del panel presenta la evidencia más consistente de alterar la función de la fenilalanina hidroxilasa, y cómo se justifica integrando información genómica, proteica y estructural?

In [6]:
# Exactamente dos variantes distintas, como pide la actividad
if len(set(SELECTED_VARIANTS)) != 2 or len(SELECTED_VARIANTS) != 2:
    raise ValueError("Seleccionen exactamente dos variantes distintas.")

# Identificadores que no aparecen en la columna ID de la tabla
missing = set(SELECTED_VARIANTS) - set(variants[ID_COLUMN])
if missing:
    raise ValueError(f"Estos identificadores no existen en la tabla: {sorted(missing)}")

# Filtra la tabla por los IDs elegidos
selected = variants[
    variants[ID_COLUMN].isin(SELECTED_VARIANTS)
].copy()

# Más de 2 filas significa que la tabla repite alguno de los IDs elegidos
if len(selected) != 2:
    raise ValueError("La columna ID tiene valores repetidos para las variantes elegidas.")

# Ordena las filas según el orden de SELECTED_VARIANTS
selected = (
    selected.set_index(ID_COLUMN, drop=False)
    .loc[SELECTED_VARIANTS]
    .reset_index(drop=True)
)

display(selected[[ID_COLUMN, PROTEIN_CHANGE_COLUMN]])

,ID,protein_change
0,VAR007,p.Gly247Asp
1,VAR003,p.Arg408Trp


# 5. Interpretar cambios missense

Se aceptan cambios en formato:
- `p.Arg408Trp`
- `p.Asp338Ala`
- `p.Ser83Leu`

Este notebook está pensado para sustituciones missense simples.

In [7]:
# Código de tres letras -> una letra
AA3_TO_1 = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C",
    "Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
    "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P",
    "Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"
}

def parse_missense(change):
    # Formato esperado: p.<aa3><posición><aa3>, p. ej. p.Arg408Trp
    # fullmatch exige que TODO el texto coincida (rechaza sufijos como "p.Arg408Trp?")
    match = re.fullmatch(
        r"p\.([A-Za-z]{3})(\d+)([A-Za-z]{3})",
        str(change)
    )

    if not match:
        raise ValueError(
            f"No es una sustitución missense simple: {change}"
        )

    ref3, pos, alt3 = match.groups()

    # Rechaza, por ejemplo, "Ter" (stop) o abreviaturas mal escritas ("ARG")
    if ref3 not in AA3_TO_1 or alt3 not in AA3_TO_1:
        raise ValueError(f"Aminoácido no reconocido: {change}")

    # Devuelve (aa original, posición base 1, aa nuevo)
    return (
        AA3_TO_1[ref3],
        int(pos),
        AA3_TO_1[alt3]
    )

# Muestra cómo se interpretó cada variante seleccionada
for _, row in selected.iterrows():
    ref, pos, alt = parse_missense(row[PROTEIN_CHANGE_COLUMN])
    print(row[ID_COLUMN], row[PROTEIN_CHANGE_COLUMN], "→", ref, pos, alt)

VAR007 p.Gly247Asp → G 247 D
VAR003 p.Arg408Trp → R 408 W


# 6. Validar contra la proteína WT

Antes de crear una mutación, comprobaremos que el residuo WT indicado realmente exista en esa posición.

Si falla:
- FASTA incorrecto;
- isoforma incorrecta;
- numeración distinta;
- anotación incorrecta;
- error en la tabla.

No continúen hasta resolverlo.

In [8]:
for _, row in selected.iterrows():
    ref, pos, alt = parse_missense(row[PROTEIN_CHANGE_COLUMN])

    # La posición está en base 1; debe caer dentro de la proteína
    if pos < 1 or pos > len(wt_seq):
        raise ValueError(
            f"{row[ID_COLUMN]}: posición {pos} fuera de rango."
        )

    # Python indexa desde 0, por eso pos - 1
    observed = wt_seq[pos - 1]

    print(
        row[ID_COLUMN],
        "| esperado:", ref,
        "| WT:", observed
    )

    # Punto de control clave: si el residuo no coincide, el FASTA o la numeración no son los correctos
    assert observed == ref, (
        f"{row[ID_COLUMN]}: se esperaba {ref}{pos}, "
        f"pero la proteína WT contiene {observed}{pos}."
    )

print("Todas las variantes coinciden con la proteína WT.")

VAR007 | esperado: G | WT: G
VAR003 | esperado: R | WT: R
Todas las variantes coinciden con la proteína WT.


# 7. Generar proteínas mutantes

In [9]:
def mutate_protein(sequence, change):
    ref, pos, alt = parse_missense(change)

    # Segunda validación del residuo WT antes de mutar
    if sequence[pos - 1] != ref:
        raise ValueError(
            f"La secuencia no contiene {ref} en la posición {pos}."
        )

    # Las cadenas son inmutables: se convierte a lista, se cambia el residuo y se reúne
    mutant = list(sequence)
    mutant[pos - 1] = alt

    return "".join(mutant)

# {ID de variante: secuencia mutante}
mutant_sequences = {}

for _, row in selected.iterrows():
    variant_id = row[ID_COLUMN]
    change = row[PROTEIN_CHANGE_COLUMN]

    mutant = mutate_protein(wt_seq, change)
    mutant_sequences[variant_id] = mutant

    # Posiciones (base 1) donde el mutante difiere del WT
    differences = [
        i + 1
        for i, (wt, mut) in enumerate(zip(wt_seq, mutant))
        if wt != mut
    ]

    print(
        variant_id,
        change,
        "| diferencias con WT:",
        differences
    )

    # Debe haber exactamente una diferencia (sustitución simple)
    assert len(differences) == 1

print("Mutantes generados correctamente.")

VAR007 p.Gly247Asp | diferencias con WT: [247]
VAR003 p.Arg408Trp | diferencias con WT: [408]
Mutantes generados correctamente.


# 8. Exportar FASTA

In [10]:
# Carpeta de salida (se crea si no existe)
output_dir = Path("rf3_inputs")
output_dir.mkdir(exist_ok=True)

def write_fasta(path, header, sequence):
    with open(path, "w") as fh:
        fh.write(f">{header}\n")
        # Líneas de 80 caracteres, el formato FASTA habitual
        for i in range(0, len(sequence), 80):
            fh.write(sequence[i:i+80] + "\n")

# Un FASTA para la proteína WT
write_fasta(
    output_dir / f"{PROTEIN_NAME}_WT.fasta",
    f"{PROTEIN_NAME}_WT",
    wt_seq
)

# Un FASTA por cada mutante, con el cambio en la cabecera
for _, row in selected.iterrows():
    variant_id = row[ID_COLUMN]
    change = row[PROTEIN_CHANGE_COLUMN]

    write_fasta(
        output_dir / f"{PROTEIN_NAME}_{variant_id}.fasta",
        f"{PROTEIN_NAME}_{variant_id}_{change}",
        mutant_sequences[variant_id]
    )

print("Archivos FASTA generados en:", output_dir)

Archivos FASTA generados en: rf3_inputs


# 9. Construir JSON para RF3

El JSON contendrá:
1. proteína WT;
2. mutante 1;
3. mutante 2.

In [11]:
# Estructura de entrada para RF3: una entrada por proteína,
# cada una con una sola cadena ("A") y su secuencia
# (formato corroborado con tutoriales y ejemplos de RF3: "name" + "components" con "seq" y "chain_id";
#  no se encontró el esquema oficial completo, confirmen campos opcionales en la documentación de foundry)
rf3_entries = [
    {
        "name": f"{PROTEIN_NAME}_WT",
        "components": [
            {
                "seq": wt_seq,
                "chain_id": "A"
            }
        ]
    }
]

for _, row in selected.iterrows():
    variant_id = row[ID_COLUMN]
    change = row[PROTEIN_CHANGE_COLUMN]

    # Quita "." y ":" del nombre del cambio para que sea un nombre seguro
    safe_change = (
        str(change)
        .replace(".", "_")
        .replace(":", "_")
    )

    rf3_entries.append({
        "name": f"{PROTEIN_NAME}_{variant_id}_{safe_change}",
        "components": [
            {
                "seq": mutant_sequences[variant_id],
                "chain_id": "A"
            }
        ]
    })

# Última expresión de la celda: Jupyter la muestra
rf3_entries

[{'name': 'PAH_WT',
  'components': [{'seq': 'MSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDVNLTHIESRPSRLKKDEYEFFTHLDKRSLPALTNIIKILRHDIGATVHELSRDKKKDTVPWFPRTIQELDRFANQILSYGAELDADHPGFKDPVYRARRKQFADIAYNYRHGQPIPRVEYMEEEKKTWGTVFKTLKSLYKTHACYEYNHIFPLLEKYCGFHEDNIPQLEDVSQFLQTCTGFRLRPVAGLLSSRDFLGGLAFRVFHCTQYIRHGSKPMYTPEPDICHELLGHVPLFSDRSFAQFSQEIGLASLGAPDEYIEKLATIYWFTVEFGLCKQGDSIKAYGAGLLSSFGELQYCLSEKPKLLPLELEKTAIQNYTVTEFQPLYYVAESFNDAKEKVRNFAATIPRPFSVRYDPYTQRIEVLDNTQQLKILADSINSEIGILCSALQKIK',
    'chain_id': 'A'}]},
 {'name': 'PAH_VAR007_p_Gly247Asp',
  'components': [{'seq': 'MSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDVNLTHIESRPSRLKKDEYEFFTHLDKRSLPALTNIIKILRHDIGATVHELSRDKKKDTVPWFPRTIQELDRFANQILSYGAELDADHPGFKDPVYRARRKQFADIAYNYRHGQPIPRVEYMEEEKKTWGTVFKTLKSLYKTHACYEYNHIFPLLEKYCGFHEDNIPQLEDVSQFLQTCTGFRLRPVADLLSSRDFLGGLAFRVFHCTQYIRHGSKPMYTPEPDICHELLGHVPLFSDRSFAQFSQEIGLASLGAPDEYIEKLATIYWFTVEFGLCKQGDSIKAYGAGLLSSFGELQYCLSEKPKLLPLELEKTAIQNYTVTEFQPLYYVAESFNDAKEKVRNFAATIPRPFSVR

# 10. Validar JSON

In [12]:
# WT + 2 mutantes = 3 entradas
assert len(rf3_entries) == 3

lengths = [
    len(entry["components"][0]["seq"])
    for entry in rf3_entries
]

print("Longitudes:", lengths)

# Las sustituciones simples no cambian la longitud: las 3 secuencias deben medir lo mismo
assert all(length > 0 for length in lengths)
assert len(set(lengths)) == 1

print("Validación correcta.")

Longitudes: [452, 452, 452]
Validación correcta.


# 11. Guardar JSON

In [13]:
json_path = output_dir / f"{PROTEIN_NAME}_RF3_inputs.json"

# indent=2 deja el JSON legible
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(rf3_entries, fh, indent=2)

print("JSON generado:", json_path)
print("\nVista previa:\n")
# Solo los primeros 1800 caracteres, para no imprimir proteínas largas completas
print(json.dumps(rf3_entries, indent=2)[:1800])

JSON generado: rf3_inputs/PAH_RF3_inputs.json

Vista previa:

[
  {
    "name": "PAH_WT",
    "components": [
      {
        "seq": "MSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDVNLTHIESRPSRLKKDEYEFFTHLDKRSLPALTNIIKILRHDIGATVHELSRDKKKDTVPWFPRTIQELDRFANQILSYGAELDADHPGFKDPVYRARRKQFADIAYNYRHGQPIPRVEYMEEEKKTWGTVFKTLKSLYKTHACYEYNHIFPLLEKYCGFHEDNIPQLEDVSQFLQTCTGFRLRPVAGLLSSRDFLGGLAFRVFHCTQYIRHGSKPMYTPEPDICHELLGHVPLFSDRSFAQFSQEIGLASLGAPDEYIEKLATIYWFTVEFGLCKQGDSIKAYGAGLLSSFGELQYCLSEKPKLLPLELEKTAIQNYTVTEFQPLYYVAESFNDAKEKVRNFAATIPRPFSVRYDPYTQRIEVLDNTQQLKILADSINSEIGILCSALQKIK",
        "chain_id": "A"
      }
    ]
  },
  {
    "name": "PAH_VAR007_p_Gly247Asp",
    "components": [
      {
        "seq": "MSTAVLENPGLGRKLSDFGQETSYIEDNCNQNGAISLIFSLKEEVGALAKVLRLFEENDVNLTHIESRPSRLKKDEYEFFTHLDKRSLPALTNIIKILRHDIGATVHELSRDKKKDTVPWFPRTIQELDRFANQILSYGAELDADHPGFKDPVYRARRKQFADIAYNYRHGQPIPRVEYMEEEKKTWGTVFKTLKSLYKTHACYEYNHIFPLLEKYCGFHEDNIPQLEDVSQFLQTCTGFRLRPVADLLSSRDFLGGLAFRVFHCTQYIRHGSKPMYTPEPD